
# AI in One Lab: End-of-Semester Survey (Expanded RL Version)

This lab covers:

- Pandas preprocessing  
- Logistic regression  
- Decision trees  
- Reinforcement learning (expanded introduction)  
- RAG-style retrieval  


In [ ]:

!pip install -q pandas scikit-learn matplotlib sentence-transformers faiss-cpu gymnasium stable-baselines3

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10,5)

from sentence_transformers import SentenceTransformer
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO


## 1. Pandas Preprocessing

In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
df.head()


In [ ]:

df = df[["Survived","Pclass","Sex","Age"]]
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male":0,"female":1})
df.head()


In [ ]:

X = df[["Pclass","Sex","Age"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape


## 2. Logistic Regression

In [ ]:

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train,y_train)
pred = log_reg.predict(X_test)
accuracy_score(y_test,pred)


In [ ]:

pd.DataFrame({"Feature":X.columns,"Weight":log_reg.coef_[0]})


## 3. Decision Tree

In [ ]:

tree = DecisionTreeClassifier(max_depth=3,random_state=42)
tree.fit(X_train,y_train)
pred_tree = tree.predict(X_test)
accuracy_score(y_test,pred_tree)


In [ ]:

plt.figure(figsize=(12,6))
plot_tree(tree, feature_names=X.columns, class_names=["Died","Survived"], filled=True)
plt.show()



# 4. Reinforcement Learning — Expanded Introduction


In [ ]:

env = gym.make("CartPole-v1")
obs, info = env.reset()
obs


### Random Agent

In [ ]:

total_reward = 0
obs, info = env.reset()

for t in range(200):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

total_reward


### RL Loop Visualization

In [ ]:

obs, info = env.reset()
total_reward = 0

for step in range(10):
    print("Step:", step)
    print("State:", obs)
    action = env.action_space.sample()
    print("Action:", action)
    obs, reward, terminated, truncated, info = env.step(action)
    print("Reward:", reward)
    print("Done:", terminated or truncated)
    print("-"*40)
    if terminated or truncated:
        break


### Load Pretrained PPO Agent

In [ ]:

!wget -q -O ppo-CartPole-v1.zip https://huggingface.co/sb3/ppo-CartPole-v1/resolve/main/ppo-CartPole-v1.zip
model = PPO.load("ppo-CartPole-v1")
"Model loaded!"


### Run Trained Agent

In [ ]:

env = gym.make("CartPole-v1")
obs, info = env.reset()
total_reward = 0

for t in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

total_reward


# 5. Mini RAG Retrieval

In [ ]:

documents = [
    "The mitochondria is the powerhouse of the cell.",
    "Python is a programming language often used for machine learning.",
    "George Washington was the first President of the United States.",
    "Reinforcement learning trains agents using rewards and punishments.",
    "Decision trees split data based on features to make predictions."
]
documents


In [ ]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(documents, normalize_embeddings=True)
doc_emb.shape


In [ ]:

def retrieve(query,k=1):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(doc_emb,q)
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

retrieve("How do agents learn?", k=2)



# 6. Reflection Questions
1. Why is preprocessing important?  
2. Logistic regression vs decision trees — when choose each?  
3. How is RL different from supervised learning?  
4. What does RAG solve?  
5. What do you want to explore more?
